# Electricity — TimeDiff (M4) + gambe profonde · spedizione Colab unica

Questo notebook **chiude il progetto su Electricity**. È diviso in due parti:

- **PARTE 1 — TimeDiff M4 (CRITICA, veloce, torch "stock")**: allena e campiona il modello **nuovo** (M4 TimeDiff) su Electricity, completando la scala **M0→M4** sul dataset principale. **Non serve GluonTS né PyTorchTS**: gira sul PyTorch già presente su Colab → niente installazioni fragili, niente downgrade di NumPy. È il pezzo che **non si può fare in locale** (torch rotto) e che ti serve davvero.
- **PARTE 2 — gambe "profonde" (OPZIONALE, costosa)**: sweep sull'orizzonte τ per M2/M3 su Electricity (E2). Usa lo **stack bloccato** (GluonTS+PyTorchTS) e quindi un **runtime pulito**: falla solo DOPO aver salvato i risultati della Parte 1. Se vuoi solo finire la scala M0→M4, **puoi saltarla.**

> ⚠️ **Verità sul free tier**: se chiudi il tab o il laptop va in sospensione, Colab gratis uccide il run dopo ~90 min. La Parte 1 di TimeDiff è **molto più veloce** del campionamento autoregressivo di TimeGrad (una sola catena inversa sull'intero blocco futuro, non passo-per-passo), quindi ha ottime chance di finire incustodita. Comunque, per sicurezza: **monta Drive**, tieni il **laptop sveglio** (su Mac, in un Terminale: `caffeinate -dimsu`), lascia **il tab aperto**. Il BLOCCO DI RECUPERO + il backup su Drive sono l'assicurazione.

# ━━━━━━━━━━  PARTE 1 — TimeDiff M4 su Electricity (critica)  ━━━━━━━━━━

## P1.1 — Verifica la GPU
Se vedi una **Tesla T4** (o simile) la GPU è attiva. Altrimenti: **Runtime > Change runtime type > GPU**, poi riesegui.

In [ ]:
!nvidia-smi

## P1.2 — Scarica il codice (branch con TimeDiff)
Clona il repo sul branch `feature/port-ladder-exchange`. Se il repo **era già clonato** (es. da un run precedente), la cella fa `git pull` e si porta all'ultimo commit — utile se avevi lanciato il notebook **prima** che TimeDiff fosse pushato. Il branch deve contenere `experiments/run_timediff.py` e `src/models/timediff.py`: la cella P1.3 lo verifica.

In [ ]:
REPO_URL = "https://github.com/Icaica14/pml-diffusion-tsf.git"
BRANCH   = "feature/port-ladder-exchange"

import os
if not os.path.isdir("/content/pml-diffusion-tsf"):
    !git clone --branch $BRANCH $REPO_URL /content/pml-diffusion-tsf
%cd /content/pml-diffusion-tsf
# se il repo era gia' clonato (run precedente), aggiorna all'ultimo commit:
!git fetch origin $BRANCH && git checkout $BRANCH && git pull --ff-only origin $BRANCH
!git log --oneline -1

## P1.3 — Guard: il codice TimeDiff c'è?
Controlla che i file nuovi di TimeDiff siano davvero sul branch clonato. Se mancano, il branch **non è ancora stato pushato** con TimeDiff: fermati e dillo a Claude (deve fare commit+push).

In [ ]:
import pathlib
_need = ["experiments/run_timediff.py", "src/models/timediff.py"]
_missing = [p for p in _need if not pathlib.Path("/content/pml-diffusion-tsf", p).is_file()]
assert not _missing, (
    "FILE TIMEDIFF MANCANTI sul branch clonato: " + ", ".join(_missing) +
    "\n-> Il branch non e' ancora stato pushato con TimeDiff. Dillo a Claude: serve commit+push."
)
print("OK: i file TimeDiff sono presenti sul branch.")

## P1.4 — Controllo ambiente (torch "stock", nessuna installazione)
TimeDiff è **PyTorch puro**: non importa GluonTS/PyTorchTS, quindi **non serve installare nulla** e **non si tocca NumPy**. Basta confermare che torch vede la GPU.

In [ ]:
%cd /content/pml-diffusion-tsf
import numpy, torch
print("numpy :", numpy.__version__)
print("torch :", torch.__version__)
print("cuda disponibile:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("\nNota: TimeDiff NON usa GluonTS/PyTorchTS. Nessun downgrade di NumPy, nessuna installazione.")

## P1.5 — (consigliato) Monta Google Drive
Salva i risultati nel **tuo** Drive, così sopravvivono al riciclo della VM. Parte un popup di autorizzazione: accettalo. Se salti questo passo, resta comunque il BLOCCO DI RECUPERO a video.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montato. Backup -> /content/drive/MyDrive/pml_electricity_backup/")

## P1.6 — Parametri del run M4
Default Electricity (gli stessi della scala M0→M3, per confronto onesto cella-per-cella). `CHUNK=256` perché con D=321 il tensore `(N,S,τ,D)` è enorme e va scorato a blocchi.

In [ ]:
DATASET       = "electricity"
CONFIG        = f"configs/data_{DATASET}.yaml"
CHUNK         = 256     # scoring a blocchi (D=321 -> tensore (N,S,tau,D) enorme)
EPOCHS        = 50      # epoche di training
SAMPLES       = 100     # traiettorie campionate per finestra
DIFF_STEPS    = 100     # passi di diffusione T
HIDDEN        = 64      # ampiezza canali del denoiser
BLOCKS        = 4       # blocchi residuali conv (DiffWave-style)
BETA          = "cosine"  # schedule beta ("cosine" o "linear")
MIXUP         = 0.5     # future-mixup (solo training); inference usa solo x_ar
SAMPLE_STEPS  = 100     # passi inversi (==T catena piena; meno = DDIM respacing)
ETA           = 1.0     # stocasticita DDIM (1.0~ancestrale, 0.0 deterministico)
PREDICT_BATCH = 16      # finestre per batch in campionamento (cap memoria)
print(f"dataset={DATASET} config={CONFIG} chunk={CHUNK} epochs={EPOCHS} samples={SAMPLES}")
print(f"diff_steps={DIFF_STEPS} hidden={HIDDEN} blocks={BLOCKS} beta={BETA} mixup={MIXUP}")
print(f"sample_steps={SAMPLE_STEPS} eta={ETA} predict_batch={PREDICT_BATCH}")

## P1.7 — SMOKE (pochi secondi): controlla che tutto fili
Mini fit→predict→score su poche finestre, **senza scrivere sulla registry**. Valida shape e finitezza delle metriche **prima** del run lungo. Se stampa `SMOKE OK`, sei pronto per P1.8.

In [ ]:
%cd /content/pml-diffusion-tsf
!python -m experiments.run_timediff --config $CONFIG --smoke --device cuda

## P1.8 — RUN vero (training + predict) + BLOCCO DI RECUPERO  ←  la cella lunga
Fai partire **questa** e poi puoi allontanarti. Allena TimeDiff, campiona, e **alla fine stampa il blocco di recupero** (riga CSV `timediff` a piena precisione + versioni + git + tempi) e fa il **backup su Drive** se montato. Quando torni: se sei ancora connesso vai a P1.9; se la VM è morta, **copia tutto il blocco di recupero** e incollalo a Claude.

In [ ]:
%cd /content/pml-diffusion-tsf
import datetime, pathlib, subprocess
_start = datetime.datetime.now(datetime.timezone.utc)
print("START (UTC):", _start.isoformat())

# ---- TRAINING + PREDICT (M4 TimeDiff su Electricity) ----
!python -m experiments.run_timediff --config $CONFIG --chunk $CHUNK --epochs $EPOCHS --samples $SAMPLES --diff-steps $DIFF_STEPS --hidden $HIDDEN --blocks $BLOCKS --beta-schedule $BETA --mixup $MIXUP --sample-steps $SAMPLE_STEPS --eta $ETA --predict-batch-size $PREDICT_BATCH --device cuda

# ---- BLOCCO DI RECUPERO: stampa tutto per ricostruire/recuperare ----
_end = datetime.datetime.now(datetime.timezone.utc)

def _sh(c):
    try:
        return subprocess.run(c, capture_output=True, text=True, shell=True).stdout.strip()
    except Exception as e:
        return "(errore: " + str(e) + ")"

B = "#" * 72
print("\n\n" + B)
print("# BLOCCO DI RECUPERO M4 TimeDiff -- se la VM si ricicla, INCOLLA TUTTO QUESTO a Claude")
print(B)
print("# inizio (UTC):", _start.isoformat())
print("# fine   (UTC):", _end.isoformat())
print("# durata (min):", round((_end - _start).total_seconds() / 60, 1))
print("# dataset:", DATASET, "| config:", CONFIG, "| chunk:", CHUNK, "| epochs:", EPOCHS, "| samples:", SAMPLES)
print("# diff_steps:", DIFF_STEPS, "| hidden:", HIDDEN, "| blocks:", BLOCKS, "| beta:", BETA, "| mixup:", MIXUP)
print("# sample_steps:", SAMPLE_STEPS, "| eta:", ETA, "| predict_batch:", PREDICT_BATCH)
print("# git HEAD:", _sh("git rev-parse HEAD"))
print("# git log :", _sh("git log --oneline -1"))
try:
    import numpy, torch
    print("# versioni: numpy", numpy.__version__, "| torch", torch.__version__)
except Exception as e:
    print("# versioni: (errore import:", e, ")")
print("# " + "-" * 70)
print("# REGISTRY (CSV grezzo, piena precisione) -- header + righe timediff:")
reg = pathlib.Path("/content/pml-diffusion-tsf/results/registry.csv")
if reg.is_file():
    _lines = reg.read_text().splitlines()
    print(_lines[0])
    _new = [_l for _l in _lines[1:] if len(_l.split(",")) > 2 and _l.split(",")[2] == "timediff"]
    for _l in _new:
        print(_l)
    print("# (" + str(len(_new)) + " righe timediff)")
    _out = "\n".join([_lines[0]] + _new) + "\n"
    pathlib.Path("/content/pml-diffusion-tsf/results/colab_new_rows.csv").write_text(_out)
    if pathlib.Path("/content/drive/MyDrive").is_dir():
        _bk = pathlib.Path("/content/drive/MyDrive/pml_electricity_backup")
        _bk.mkdir(parents=True, exist_ok=True)
        (_bk / "registry.csv").write_text(reg.read_text())
        (_bk / "colab_new_rows.csv").write_text(_out)
        print("# BACKUP su Drive OK: /content/drive/MyDrive/pml_electricity_backup/")
    else:
        print("# (Drive non montato: nessun backup su Drive)")
else:
    print("# !! results/registry.csv NON trovata: il run e' forse fallito prima di scrivere.")
print(B)
print("# FINE BLOCCO DI RECUPERO")
print(B)

## P1.9 — Porta a casa il risultato (se sei sveglio e connesso)
Scarica il CSV con la riga `timediff`. (Se hai montato Drive, una copia è già lì.)

In [ ]:
from google.colab import files
files.download("/content/pml-diffusion-tsf/results/colab_new_rows.csv")

## P1.10 — Ablazione ε (secondo run TimeDiff, ~40 min) — la *cura* del collasso

Il run x0 (P1.8) produce un M4 con l'**incertezza collassata**: copertura ≈ 0 e CRPS ≈ MAE (massa di probabilità degenere). Questo secondo run cambia **una sola cosa** — il bersaglio del denoiser passa da `x0` (il segnale pulito) a `ε` (il rumore iniettato, Ho et al. 2020) — con **tutti gli altri iperparametri identici**. Predicendo il rumore, l'ampiezza dei campioni è legata allo schedule e non può collassare: è l'ablazione che dovrebbe **ricalibrare** la copertura. Si registra come modello **`timediff_eps`** (M4ε), così **convive** con la riga x0 nella registry senza sovrascriverla.

> **Quando lanciarla.** Stesso ambiente della Parte 1 (torch "stock", niente GluonTS): **nessun restart**. Due modi:
> - **subito dopo P1.8/P1.9**, nella stessa sessione; **oppure**
> - in una **sessione fresca**: esegui P1.1→P1.7 (setup + smoke), poi **salta P1.8/P1.9** (il run x0 è già bancato) e vieni qui.
>
> Riusa i **parametri di P1.6** (`EPOCHS`, `SAMPLES`, …): un'ablazione pulita cambia solo `--param eps`. Mixup resta a 0.5 come l'x0 — così l'unica variabile è la parameterizzazione (`mixup=0.0` è un'ulteriore ablazione opzionale, non questa).

In [ ]:
%cd /content/pml-diffusion-tsf
import datetime, pathlib, subprocess
_start = datetime.datetime.now(datetime.timezone.utc)
print("START eps (UTC):", _start.isoformat())

# ---- TRAINING + PREDICT (M4e TimeDiff eps-prediction su Electricity) ----
# Iperparametri IDENTICI a P1.8; cambia SOLO --param eps -> modello 'timediff_eps'.
!python -m experiments.run_timediff --config $CONFIG --param eps --chunk $CHUNK --epochs $EPOCHS --samples $SAMPLES --diff-steps $DIFF_STEPS --hidden $HIDDEN --blocks $BLOCKS --beta-schedule $BETA --mixup $MIXUP --sample-steps $SAMPLE_STEPS --eta $ETA --predict-batch-size $PREDICT_BATCH --device cuda

# ---- BLOCCO DI RECUPERO eps: stampa tutto per ricostruire/recuperare ----
_end = datetime.datetime.now(datetime.timezone.utc)

def _sh(c):
    try:
        return subprocess.run(c, capture_output=True, text=True, shell=True).stdout.strip()
    except Exception as e:
        return "(errore: " + str(e) + ")"

B = "#" * 72
print("\n\n" + B)
print("# BLOCCO DI RECUPERO M4eps (timediff_eps) -- se la VM si ricicla, INCOLLA TUTTO a Claude")
print(B)
print("# inizio (UTC):", _start.isoformat())
print("# fine   (UTC):", _end.isoformat())
print("# durata (min):", round((_end - _start).total_seconds() / 60, 1))
print("# param: eps | dataset:", DATASET, "| config:", CONFIG, "| chunk:", CHUNK, "| epochs:", EPOCHS, "| samples:", SAMPLES)
print("# diff_steps:", DIFF_STEPS, "| hidden:", HIDDEN, "| blocks:", BLOCKS, "| beta:", BETA, "| mixup:", MIXUP)
print("# sample_steps:", SAMPLE_STEPS, "| eta:", ETA, "| predict_batch:", PREDICT_BATCH)
print("# git HEAD:", _sh("git rev-parse HEAD"))
print("# git log :", _sh("git log --oneline -1"))
try:
    import numpy, torch
    print("# versioni: numpy", numpy.__version__, "| torch", torch.__version__)
except Exception as e:
    print("# versioni: (errore import:", e, ")")
print("# " + "-" * 70)
print("# REGISTRY (CSV grezzo, piena precisione) -- header + righe timediff_eps:")
reg = pathlib.Path("/content/pml-diffusion-tsf/results/registry.csv")
if reg.is_file():
    _lines = reg.read_text().splitlines()
    print(_lines[0])
    _new = [_l for _l in _lines[1:] if len(_l.split(",")) > 2 and _l.split(",")[2] == "timediff_eps"]
    for _l in _new:
        print(_l)
    print("# (" + str(len(_new)) + " righe timediff_eps)")
    _out = "\n".join([_lines[0]] + _new) + "\n"
    pathlib.Path("/content/pml-diffusion-tsf/results/colab_new_rows_eps.csv").write_text(_out)
    if pathlib.Path("/content/drive/MyDrive").is_dir():
        _bk = pathlib.Path("/content/drive/MyDrive/pml_electricity_backup")
        _bk.mkdir(parents=True, exist_ok=True)
        (_bk / "registry.csv").write_text(reg.read_text())
        (_bk / "colab_new_rows_eps.csv").write_text(_out)
        print("# BACKUP su Drive OK: /content/drive/MyDrive/pml_electricity_backup/")
    else:
        print("# (Drive non montato: nessun backup su Drive)")
else:
    print("# !! results/registry.csv NON trovata: il run e' forse fallito prima di scrivere.")
print(B)
print("# FINE BLOCCO DI RECUPERO eps")
print(B)

In [ ]:
from google.colab import files
files.download("/content/pml-diffusion-tsf/results/colab_new_rows_eps.csv")

# ━━━━━━━━━━  PARTE 2 — gambe "profonde" su Electricity (OPZIONALE)  ━━━━━━━━━━

> 🛑 **FERMATI E LEGGI.** Questa parte è **facoltativa** e **costosa**, e usa lo **stack bloccato** (GluonTS 0.13 + PyTorchTS + NumPy<2). Quello stack **è incompatibile** con il torch "stock" della Parte 1.
>
> **Procedura corretta:**
> 1. Assicurati di aver **scaricato/sottomesso** i risultati della Parte 1 (P1.9 o BLOCCO DI RECUPERO).
> 2. **Runtime > Disconnetti ed elimina runtime** (runtime pulito), poi riconnetti la GPU.
> 3. Esegui **solo** le celle della Parte 2, in ordine, ripartendo da P2.1.
>
> Cosa fa: **sweep sull'orizzonte τ** per M2 DeepAR e M3 TimeGrad su Electricity (E2), che alimenta `figures/presentation/fig_e2_horizon.png`. Ogni τ rifà training+predict: su Electricity sono **diverse ore**. Se vuoi solo finire la scala M0→M4, **salta tutta la Parte 2.**

## P2.1 — (runtime pulito) Clona + stack bloccato
A fine cella: **Runtime > Restart session**, poi riparti da **P2.2** (NON rifare questa).

In [ ]:
REPO_URL = "https://github.com/Icaica14/pml-diffusion-tsf.git"
BRANCH   = "feature/port-ladder-exchange"
import os
if not os.path.isdir("/content/pml-diffusion-tsf"):
    !git clone --branch $BRANCH $REPO_URL /content/pml-diffusion-tsf
%cd /content/pml-diffusion-tsf
!pip install -q "gluonts[torch]==0.13.7" "numpy<2" "pandas<2.2"
!pip install -q --no-deps "git+https://github.com/zalandoresearch/pytorch-ts.git@81be06bcc"
print("\nStack bloccato installato. Ora: Runtime > Restart session, poi vai a P2.2.")

## P2.2 — Controllo ambiente (dopo il restart)
Se vedi `cuda: True` e gli import passano, sei pronto.

In [ ]:
%cd /content/pml-diffusion-tsf
import numpy, pandas, torch, gluonts, pts
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| torch", torch.__version__, "| gluonts", gluonts.__version__)
print("cuda:", torch.cuda.is_available())

## P2.3 — (consigliato) Monta Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## P2.4 — E2 sweep orizzonte (M2 DeepAR + M3 TimeGrad) su Electricity + recupero
Griglia τ = 12,24,48,96. Ogni punto rifà training+predict: **diverse ore**. Scrive su `results/sweeps/horizon.csv`. Alla fine stampa il blocco da incollare a Claude.

In [ ]:
%cd /content/pml-diffusion-tsf
import datetime, pathlib
_start = datetime.datetime.now(datetime.timezone.utc)
print("START (UTC):", _start.isoformat())

!python -m experiments.run_horizon_sweep --models deepar,timegrad --config configs/data_electricity.yaml --taus 12,24,48,96 --chunk 256 --epochs 50 --samples 100 --device cuda

_end = datetime.datetime.now(datetime.timezone.utc)
B = "#" * 72
print("\n\n" + B)
print("# BLOCCO DI RECUPERO E2 (horizon sweep) -- INCOLLA TUTTO QUESTO a Claude")
print(B)
print("# durata (min):", round((_end - _start).total_seconds() / 60, 1))
sw = pathlib.Path("/content/pml-diffusion-tsf/results/sweeps/horizon.csv")
if sw.is_file():
    print("# results/sweeps/horizon.csv (grezzo):")
    print(sw.read_text())
    if pathlib.Path("/content/drive/MyDrive").is_dir():
        _bk = pathlib.Path("/content/drive/MyDrive/pml_electricity_backup"); _bk.mkdir(parents=True, exist_ok=True)
        (_bk / "horizon.csv").write_text(sw.read_text())
        print("# BACKUP su Drive OK")
else:
    print("# !! results/sweeps/horizon.csv NON trovata.")
print(B)

## P2.5 — Scarica lo sweep

In [ ]:
from google.colab import files
files.download("/content/pml-diffusion-tsf/results/sweeps/horizon.csv")

# E adesso?

**Parte 1 — x0 (M4 TimeDiff):** mandami `colab_new_rows.csv` **oppure** incolla il BLOCCO DI RECUPERO di P1.8. Unisco la riga `timediff` Electricity alla registry e, in locale, rigenero tabella + figure (ora **M0→M4**) e propago i numeri nel deck/report.

**Parte 1 — ε (ablazione, M4ε):** se hai fatto anche **P1.10**, mandami `colab_new_rows_eps.csv` **oppure** il BLOCCO DI RECUPERO eps. Banco la riga `timediff_eps`, rigenero tabella + figure (M4 **x0 vs ε** affiancati) e scrivo l'arco del report: *x0 collassa → analisi della varianza → ε ricalibra*.

**Parte 2 (E2 sweep), se l'hai fatta:** mandami anche `horizon.csv` (o il blocco di P2.4): aggiorno `results/sweeps/horizon.csv` e ridisegno `fig_e2_horizon.png`.

Se un run si è interrotto a metà (guarda a che epoca era arrivato nel log), lo rilanciamo — niente resume, purtroppo. Poi **committo io** (solo dopo tuo via libera).